# 1단계 워크플로우 vs 일반 요청 분류 모델 학습

이 노트북은 KLUE-RoBERTa와 SetFit 모델을 사용하여 워크플로우 요청과 일반 작업 요청을 분류하는 모델을 학습합니다.

## 데이터셋
- 총 샘플: 801개
- 워크플로우: 471개 (58.8%)
- 일반 작업: 330개 (41.2%)
- 학습/검증/테스트: 640/80/81


## 1. 환경 설정


In [ ]:
# 필수 라이브러리 설치
%pip install -q transformers datasets accelerate setfit scikit-learn pandas numpy torch torchvision torchaudio
%pip install -q sentence-transformers


In [ ]:
# GPU/TPU 확인
import torch
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU 메모리: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# TPU 확인
try:
    import torch_xla.core.xla_model as xm
    print(f"TPU 사용 가능: True")
    print(f"TPU 디바이스: {xm.xla_device()}")
except ImportError:
    print("TPU 사용 불가 (torch_xla 미설치)")


## 2. 데이터셋 로드

**데이터셋 업로드 방법:**
1. 코랩 파일 업로드: 왼쪽 파일 아이콘 → 업로드 → train.csv, val.csv, test.csv 업로드
2. Google Drive 마운트: 아래 코드의 주석 해제
3. GitHub에서 다운로드: 아래 코드의 주석 해제


In [ ]:
# Google Drive 마운트 (데이터셋이 Drive에 있는 경우)
from google.colab import drive
drive.mount('/content/drive')

# 또는 직접 업로드한 파일 사용
import pandas as pd
import json
from pathlib import Path

# 방법 1: 코랩에 직접 업로드한 파일 사용
# train_df = pd.read_csv('/content/train.csv')
# val_df = pd.read_csv('/content/val.csv')
# test_df = pd.read_csv('/content/test.csv')

# 방법 2: Google Drive에서 로드
# train_df = pd.read_csv('/content/drive/MyDrive/datasets/step1_classification_dataset/train.csv')
# val_df = pd.read_csv('/content/drive/MyDrive/datasets/step1_classification_dataset/val.csv')
# test_df = pd.read_csv('/content/drive/MyDrive/datasets/step1_classification_dataset/test.csv')

# 방법 3: GitHub에서 다운로드 (실제 저장소 URL로 변경 필요)
# !wget https://raw.githubusercontent.com/your-repo/ai_module/main/data/step1_classification_dataset/train.csv
# !wget https://raw.githubusercontent.com/your-repo/ai_module/main/data/step1_classification_dataset/val.csv
# !wget https://raw.githubusercontent.com/your-repo/ai_module/main/data/step1_classification_dataset/test.csv
# train_df = pd.read_csv('train.csv')
# val_df = pd.read_csv('val.csv')
# test_df = pd.read_csv('test.csv')

# 방법 4: 샘플 데이터로 테스트 (실제 데이터셋이 없을 때)
print("샘플 데이터 생성 중...")
train_data = [
    {"query": "다음 주 3일간 제주도로 워크샵 다녀올게", "label": 1},
    {"query": "12월 20일부터 22일까지 부산 출장 갈게", "label": 1},
    {"query": "내일부터 3일간 휴가 신청하고 싶어", "label": 1},
    {"query": "출장 신청해줘", "label": 1},
    {"query": "휴가 다녀올게", "label": 1},
    {"query": "출장 일정 취소해줘", "label": 0},
    {"query": "출장 규정 알려줘", "label": 0},
    {"query": "휴가 신청 방법은?", "label": 0},
    {"query": "문서 작성해줘", "label": 0},
    {"query": "오늘 식단 뭐야?", "label": 0},
]

# 샘플 데이터로 DataFrame 생성 (실제 데이터셋 사용 시 주석 처리)
train_df = pd.DataFrame(train_data * 80)  # 800개 샘플
val_df = pd.DataFrame(train_data[:5] * 10)  # 50개 샘플
test_df = pd.DataFrame(train_data[5:] * 10)  # 50개 샘플

print(f"학습 세트: {len(train_df)}개")
print(f"검증 세트: {len(val_df)}개")
print(f"테스트 세트: {len(test_df)}개")
print(f"\n클래스 분포 (학습 세트):")
print(train_df['label'].value_counts())
print(f"\n워크플로우: {train_df['label'].sum()}개 ({train_df['label'].sum()/len(train_df)*100:.1f}%)")
print(f"일반 작업: {(train_df['label']==0).sum()}개 ({(train_df['label']==0).sum()/len(train_df)*100:.1f}%)")


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import os

# 모델 및 토크나이저 로드
model_name = "klue/roberta-base"
print(f"모델 로드 중: {model_name}")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    problem_type="single_label_classification"
)

print("모델 로드 완료")
print(f"모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
# 데이터셋 준비
def tokenize_function(examples):
    return tokenizer(
        examples["query"],
        truncation=True,
        padding=True,
        max_length=128
    )

# DataFrame을 Dataset으로 변환
train_dataset = Dataset.from_pandas(train_df[["query", "label"]])
val_dataset = Dataset.from_pandas(val_df[["query", "label"]])
test_dataset = Dataset.from_pandas(test_df[["query", "label"]])

# 토크나이징
print("데이터셋 토크나이징 중...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print("데이터셋 전처리 완료")


In [ ]:
# 평가 메트릭 정의
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )
    
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


In [ ]:
# 학습 설정
output_dir = "./step1_klue_roberta_model"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    logging_dir=f"{output_dir}/logs",
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=3,
    fp16=torch.cuda.is_available(),  # GPU가 있으면 mixed precision 사용
    dataloader_num_workers=2,
    report_to="none",  # wandb 등 로깅 비활성화
)

# Trainer 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("학습 시작...")
print(f"학습 샘플 수: {len(train_dataset)}")
print(f"검증 샘플 수: {len(val_dataset)}")
print(f"배치 크기: {training_args.per_device_train_batch_size}")
print(f"총 스텝 수: {len(train_dataset) // training_args.per_device_train_batch_size * training_args.num_train_epochs}")

trainer.train()

print("\n학습 완료!")


In [ ]:
# 테스트 세트 평가
print("테스트 세트 평가 중...")
test_results = trainer.evaluate(test_dataset)

print("\n=== 테스트 세트 성능 ===")
for key, value in test_results.items():
    if 'loss' not in key:
        print(f"{key}: {value:.4f}")

# 예측 및 혼동 행렬
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = test_df['label'].values

print("\n=== 혼동 행렬 ===")
cm = confusion_matrix(true_labels, pred_labels)
print(cm)
print("\n(행: 실제, 열: 예측)")
print("0: 일반 작업, 1: 워크플로우")


In [ ]:
# 모델 저장
final_model_path = "./step1_klue_roberta_final"
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"모델 저장 완료: {final_model_path}")

# Google Drive에 저장 (선택적)
# !cp -r {final_model_path} /content/drive/MyDrive/models/

# 또는 다운로드
# from google.colab import files
# !zip -r model.zip {final_model_path}
# files.download('model.zip')


## 4. SetFit 모델 학습


In [ ]:
from setfit import SetFitModel, SetFitTrainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# SetFit 모델 초기화 (다국어 지원)
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
print(f"SetFit 모델 로드 중: {model_name}")

setfit_model = SetFitModel.from_pretrained(model_name)
print("모델 로드 완료")


In [ ]:
# SetFit용 데이터셋 준비 (토크나이징 불필요)
setfit_train_dataset = Dataset.from_pandas(train_df[["query", "label"]])
setfit_val_dataset = Dataset.from_pandas(val_df[["query", "label"]])
setfit_test_dataset = Dataset.from_pandas(test_df[["query", "label"]])

print("데이터셋 준비 완료")


In [ ]:
# SetFit 학습 설정
trainer = SetFitTrainer(
    model=setfit_model,
    train_dataset=setfit_train_dataset,
    eval_dataset=setfit_val_dataset,
    num_epochs=1,  # SetFit은 보통 1 epoch로 충분
    batch_size=16,
    learning_rate=2e-5,
    num_iterations=20,  # Contrastive learning 반복 횟수
    column_mapping={"query": "text", "label": "label"},  # 컬럼 매핑
)

print("SetFit 학습 시작...")
print(f"학습 샘플 수: {len(setfit_train_dataset)}")
print(f"검증 샘플 수: {len(setfit_val_dataset)}")

trainer.train()

print("\n학습 완료!")


In [ ]:
# 검증 세트 평가
val_metrics = trainer.evaluate()
print("\n=== 검증 세트 성능 ===")
for key, value in val_metrics.items():
    print(f"{key}: {value:.4f}")

# 테스트 세트 평가
test_predictions = setfit_model.predict([item["query"] for item in setfit_test_dataset])
test_labels = test_df['label'].values

accuracy = accuracy_score(test_labels, test_predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels, test_predictions, average='weighted'
)

print("\n=== 테스트 세트 성능 ===")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


In [ ]:
# SetFit 모델 저장
setfit_model_path = "./step1_setfit_model"
trainer.model.save_pretrained(setfit_model_path)

print(f"SetFit 모델 저장 완료: {setfit_model_path}")

# Google Drive에 저장 (선택적)
# !cp -r {setfit_model_path} /content/drive/MyDrive/models/

# 또는 다운로드
# from google.colab import files
# !zip -r setfit_model.zip {setfit_model_path}
# files.download('setfit_model.zip')


## 5. 모델 추론 테스트


In [ ]:
# KLUE-RoBERTa 모델 추론
from transformers import pipeline

# 저장된 모델 로드
classifier = pipeline(
    "text-classification",
    model=final_model_path,
    tokenizer=final_model_path,
    device=0 if torch.cuda.is_available() else -1
)

# 테스트 쿼리
test_queries = [
    "다음 주 3일간 제주도로 워크샵 다녀올게",  # 워크플로우
    "출장 일정 취소해줘",  # 일반 작업
    "출장 신청해줘",  # 워크플로우
    "문서 작성해줘",  # 일반 작업
    "12월 20일부터 22일까지 부산 출장 갈게",  # 워크플로우
    "출장 규정 알려줘",  # 일반 작업
]

print("=== KLUE-RoBERTa 추론 결과 ===")
for query in test_queries:
    result = classifier(query)[0]
    label = "워크플로우" if result['label'] == 'LABEL_1' else "일반 작업"
    print(f"\n쿼리: {query}")
    print(f"예측: {label} (신뢰도: {result['score']:.4f})")


In [ ]:
# SetFit 모델 추론
from setfit import SetFitModel

# 저장된 모델 로드
loaded_setfit_model = SetFitModel.from_pretrained(setfit_model_path)

print("=== SetFit 추론 결과 ===")
for query in test_queries:
    prediction = loaded_setfit_model.predict([query])[0]
    proba = loaded_setfit_model.predict_proba([query])[0]
    label = "워크플로우" if prediction == 1 else "일반 작업"
    confidence = proba[prediction]
    print(f"\n쿼리: {query}")
    print(f"예측: {label} (신뢰도: {confidence:.4f})")


## 6. 학습 요약

### 모델 비교

| 모델 | 학습 시간 | 정확도 | 장점 | 단점 |
|------|----------|--------|------|------|
| KLUE-RoBERTa | 18-30분 (TPU) / 21-37분 (GPU) | 높음 | 높은 정확도, 문맥 이해 우수 | 학습 시간 길음 |
| SetFit | 4-10분 (GPU) | 중간 | 빠른 학습, 작은 데이터셋에 적합 | 정확도 상대적으로 낮을 수 있음 |

### 다음 단계
1. 학습된 모델을 다운로드
2. 프로젝트에 통합 (`orchestrator_v3.py`)
3. 성능 모니터링 및 개선
